In [1]:
# %% Imports
import jax
from jax import numpy as jnp
from jax import random as jr

In [2]:
# %% Define NB metric
def new_metric(x, y):
    dim = x.shape[0]

    def single_dim(x, y):
        num = jnp.abs(x - y) * (jnp.abs(x) + jnp.abs(y))
        den = x**2 + y**2
        # avoid 0/0 -> NaN
        return jnp.where(den != 0, 0.5 * (num / den), 0.0)

    return jax.lax.fori_loop(0, dim, lambda i, acc: acc + single_dim(x[i], y[i]), 0.0)


In [3]:
nmet = jax.vmap(new_metric, in_axes=(1, 1))

# %% Test the metrics
N = 10_000
key = jr.key(23)
key, *subkey = jr.split(key, 4)

x = jr.uniform(subkey[0], (2, N), minval=-1, maxval=1)
y = jr.uniform(subkey[1], (2, N), minval=-1, maxval=1)
print(nmet(x, y))


[0.43644905 1.5626042  1.1756217  ... 1.3585368  0.8820763  0.6918911 ]


In [4]:
# %% Check metric conditions
# Symmetry (tolerant for floats)
print(jnp.allclose(nmet(x, y), nmet(y, x), rtol=1e-6, atol=1e-8))

True


In [5]:
# Identity (d(x,x)=0)
print(jnp.allclose(nmet(x, x), 0.0, rtol=0.0, atol=1e-8))

True


In [6]:
# Triangle inequality (make z same range as x,y)
z = jr.uniform(subkey[2], (2, N), minval=-1, maxval=1)
print(jnp.all(nmet(x, z) <= nmet(x, y) + nmet(y, z) + 1e-8))

True


In [10]:
# Triangle inequality (make z same range as x,y)
z = jr.uniform(subkey[2], (2, N), minval=-1, maxval=1)

dxz = nmet(x, z)
dxy = nmet(x, y)
dyz = nmet(y, z)

tri_ok = jnp.all(dxz <= dxy + dyz + 1e-8)
print(tri_ok)

# --- NEW: worst-case triangle violation report ---
viol = dxz - (dxy + dyz)              # elementwise violation amount
max_viol = jnp.max(viol)              # worst violation
argmax_viol = jnp.argmax(viol)        # index of worst case

print("max triangle violation (dxz - dxy - dyz):", max_viol)
print("worst-case index:", argmax_viol)

# optional: show the actual values at the worst case
k = int(argmax_viol)
print("worst-case values: dxz, dxy, dyz =", dxz[k], dxy[k], dyz[k])

# optional: how many (if any) strict violations beyond tolerance?
tol = 1e-8
num_viol = jnp.sum(viol > tol)
print("count of violations >", tol, ":", num_viol)


True
max triangle violation (dxz - dxy - dyz): -0.0025261939
worst-case index: 6078
worst-case values: dxz, dxy, dyz = 0.3816255 0.24271826 0.14143342
count of violations > 1e-08 : 0


In [7]:
# Positivity:
# 1) non-negativity (safer/basic)
print(jnp.all(nmet(x, y) >= -1e-12))

True


In [8]:
# 2) strict positivity for x!=y (mask equal samples just in case)
eq_mask = jnp.all(x == y, axis=0)
dxy = nmet(x, y)
print(jnp.all(dxy[~eq_mask] > 0.0))

True


In [9]:
# also check x vs -x (should be >0 except when x==0 vector)
dx_negx = nmet(x, -x)
zero_mask = jnp.all(x == 0, axis=0)
print(jnp.all(dx_negx[~zero_mask] > 0.0))

True
